<a href="https://colab.research.google.com/github/VamuveTV/3DTrajMaster/blob/main/GugaMultipleVoice2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install SpeechBrain**

In [1]:
# --- STEP 1: CONSOLIDATED FIX (USING CORRECT VERSION 2.5.1) ---

# 1. Force uninstall all conflicting packages to ensure a clean slate.
!pip uninstall -y torch torchvision torchaudio speechbrain librosa soundfile

# 2. Install the known compatible PyTorch stack (v2.5.1 for CUDA 12.1).
# This version is confirmed to be available and contains the required function.
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

# 3. Reinstall remaining dependencies.
!pip install speechbrain librosa soundfile

# Final confirmation message
print("\n************************************************************************")
print("  ✅ INSTALLATION COMPLETED SUCCESSFULLY. PROCEED TO STEP 2.")
print("************************************************************************")

Found existing installation: torch 2.9.0+cu126
Uninstalling torch-2.9.0+cu126:
  Successfully uninstalled torch-2.9.0+cu126
Found existing installation: torchvision 0.24.0+cu126
Uninstalling torchvision-0.24.0+cu126:
  Successfully uninstalled torchvision-0.24.0+cu126
Found existing installation: torchaudio 2.9.0+cu126
Uninstalling torchaudio-2.9.0+cu126:
  Successfully uninstalled torchaudio-2.9.0+cu126
Found existing installation: librosa 0.11.0
Uninstalling librosa-0.11.0:
  Successfully uninstalled librosa-0.11.0
Found existing installation: soundfile 0.13.1
Uninstalling soundfile-0.13.1:
  Successfully uninstalled soundfile-0.13.1
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 125.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 118.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 M

**Mount Google Drive**

In [2]:
from google.colab import drive
import os
import io
from pathlib import Path

drive.mount('/gdrive')

Mounted at /gdrive


**Define Configuration**

In [3]:
# Customize the following options!
# This is the path to the 2-speaker separation model (Sepformer).
# separation_model_path = "speechbrain/sepformer-wsj02mix"
# separation_model_path = "speechbrain/sepformer-wham16k"   # ← change this line only
separation_model_path = "speechbrain/sepformer-whamr16k"  # ← Use this exact name
# File types to look for
extensions = ["mp3", "wav", "ogg", "flac"]

# Output options
mp3 = True        # Convert final output files to MP3? (Saves space)
mp3_rate = 320    # MP3 bitrate in kbps (320 is high quality)

# --- YOUR CUSTOM PATHS BELOW ---
# The folder where your audio files (e.g., meeting.mp3) are located.
in_path = '/gdrive/MyDrive/demucs/'

# The folder where the separated voice files will be saved.
out_path = '/gdrive/MyDrive/voice_separated/'

**Define Separation Functions**

In [4]:
#@title Useful functions, don't forget to execute
import subprocess as sp
import sys
import torch
torch.cuda.empty_cache()
import gc
gc.collect()
import torchaudio
import soundfile as sf
from speechbrain.inference.separation import SepformerSeparation

def find_files(in_path):
    out = []
    for item in os.listdir(in_path):
        file = Path(in_path) / item
        if file.suffix.lower().lstrip(".") in extensions:
            out.append(file)
    return out

# Initialize the separation model once
try:
    # Load the Sepformer model for two-speaker separation
    device = "cuda" if torch.cuda.is_available() else "cpu"
    separator = SepformerSeparation.from_hparams(source=separation_model_path,
                                                 run_opts={"device":device})
    print(f"SpeechBrain Sepformer model loaded successfully on {device}.")
except Exception as e:
    print(f"Error loading SpeechBrain model: {e}")
    separator = None

def separate_voices(inp=None, outp=None):
    if separator is None:
        print("Cannot run separation: Model failed to load.")
        return

    inp = inp or in_path
    outp = outp or out_path
    Path(outp).mkdir(parents=True, exist_ok=True)

    files_to_process = find_files(inp)
    if not files_to_process:
        print(f"No valid audio files in {inp}")
        return

    print("Going to separate the files:")
    print('\n'.join(map(str, files_to_process)))

    for file_path in files_to_process:
        print(f"\nProcessing: {file_path.name}")

        try:
            # THIS IS THE KEY CHANGE: use separate_file instead of loading everything yourself
            est_sources = separator.separate_file(str(file_path))

            # est_sources shape: (sources, samples, channels) → usually (2, samples, 1)
            est_sources = est_sources.squeeze(-1)  # remove channel dim if present

            base_name = file_path.stem
            output_folder = Path(outp) / base_name
            output_folder.mkdir(parents=True, exist_ok=True)

            print(f"Found {est_sources.shape[0]} separated streams. Saving to: {output_folder}")

            for i in range(est_sources.shape[0]):
                wav_file = output_folder / f"speaker_{i+1}.wav"
                sf.write(str(wav_file), est_sources[i].cpu().numpy().T, separator.sample_rate)

                if mp3:
                    mp3_file = output_folder / f"speaker_{i+1}.mp3"
                    cmd = ["ffmpeg", "-i", str(wav_file), "-b:a", f"{mp3_rate}k", "-y", str(mp3_file)]
                    sp.run(cmd, stdout=sp.PIPE, stderr=sp.PIPE, check=True)
                    print(f" - Saved speaker {i+1} as MP3: {mp3_file.name}")
                    os.remove(wav_file)  # clean up WAV
                else:
                    print(f" - Saved speaker {i+1} as WAV: {wav_file.name}")

            # Clean up memory after each file
            del est_sources
            torch.cuda.empty_cache()

        except RuntimeError as e:
            if "CUDA out of memory" in str(e):
                print(f"Separation failed for {file_path.name}: CUDA out of memory (even with chunking)")
                print("   → Try splitting this file into shorter parts first (<2–3 min each)")
            else:
                print(f"Separation failed for {file_path.name}: {e}")
        except Exception as e:
            print(f"Separation failed for {file_path.name}: {e}")

DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _speechbrain_save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _speechbrain_load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _recover
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/sepformer-whamr16k' if not cached


hyperparams.yaml: 0.00B [00:00, ?B/s]

INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/sepformer-whamr16k' if not cached
DEBUG:speechbrain.utils.parameter_transfer:Fetching files for pretraining (no collection directory set)
INFO:speechbrain.utils.fetching:Fetch masknet.ckpt: Fetching from HuggingFace Hub 'speechbrain/sepformer-whamr16k' if not cached


masknet.ckpt:   0%|          | 0.00/113M [00:00<?, ?B/s]

DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["masknet"] = /root/.cache/huggingface/hub/models--speechbrain--sepformer-whamr16k/snapshots/21a5b500c6f52fddc387c5d9e5fb13ffd6f039c5/masknet.ckpt
INFO:speechbrain.utils.fetching:Fetch encoder.ckpt: Fetching from HuggingFace Hub 'speechbrain/sepformer-whamr16k' if not cached


encoder.ckpt:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["encoder"] = /root/.cache/huggingface/hub/models--speechbrain--sepformer-whamr16k/snapshots/21a5b500c6f52fddc387c5d9e5fb13ffd6f039c5/encoder.ckpt
INFO:speechbrain.utils.fetching:Fetch decoder.ckpt: Fetching from HuggingFace Hub 'speechbrain/sepformer-whamr16k' if not cached


decoder.ckpt:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["decoder"] = /root/.cache/huggingface/hub/models--speechbrain--sepformer-whamr16k/snapshots/21a5b500c6f52fddc387c5d9e5fb13ffd6f039c5/decoder.ckpt
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: masknet, encoder, decoder
DEBUG:speechbrain.utils.parameter_transfer:Redirecting (loading from local path): masknet -> /root/.cache/huggingface/hub/models--speechbrain--sepformer-whamr16k/snapshots/21a5b500c6f52fddc387c5d9e5fb13ffd6f039c5/masknet.ckpt
DEBUG:speechbrain.utils.parameter_transfer:Redirecting (loading from local path): encoder -> /root/.cache/huggingface/hub/models--speechbrain--sepformer-whamr16k/snapshots/21a5b500c6f52fddc387c5d9e5fb13ffd6f039c5/encoder.ckpt
DEBUG:speechbrain.utils.parameter_transfer:Redirecting (loading from local path): decoder -> /root/.cache/huggingface/hub/models--speechbrain--sepformer-whamr16k/snapshots/21a5b500c6f52fddc387c5d9e5fb13ffd6f039c5/decoder.ckpt
/usr/l

SpeechBrain Sepformer model loaded successfully on cuda.


**Run Separation**

In [11]:
# EXECUTE THIS TO START SEPARATION ON YOUR GOOGLE DRIVE FILES
separate_voices()



Going to separate the files:
/gdrive/MyDrive/demucs/Guga_UlotraSeven_Sobreposto_Amarildo.mp3

Processing: Guga_UlotraSeven_Sobreposto_Amarildo.mp3
Resampling the audio from 48000 Hz to 16000 Hz
Separation failed for Guga_UlotraSeven_Sobreposto_Amarildo.mp3: CUDA out of memory (even with chunking)
   → Try splitting this file into shorter parts first (<2–3 min each)


In [9]:
from pathlib import Path
import subprocess as sp
import soundfile as sf
import torch
import torchaudio
import numpy as np
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import gc


# Folders
in_path = '/gdrive/MyDrive/demucs/'
out_path = '/gdrive/MyDrive/voice_separated/'
Path(out_path).mkdir(parents=True, exist_ok=True)

# Clear GPU memory
torch.cuda.empty_cache()
gc.collect()

print("Loading Sepformer WHAMR! 16kHz model (2 speakers + noise/reverb)...")
separator = SepformerSeparation.from_hparams(
    source="speechbrain/sepformer-whamr16k",
    savedir="sepformer_whamr16k_cache",
    run_opts={"device": "cuda" if torch.cuda.is_available() else "cpu"}
)
print("Model loaded!")

def separate_with_chunks(file_path, chunk_length_sec=30, overlap_sec=3.0):
    """
    Process long audio by chunking, separate each chunk, then overlap-add.
    chunk_length_sec=60 is conservative → adjust to 90–120 if your GPU allows.
    """
    waveform, orig_sr = torchaudio.load(file_path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)  # to mono
    model_sr = separator.hparams.sample_rate  # 16000

    # Resample to model rate
    if orig_sr != model_sr:
        print(f"Resampling from {orig_sr} Hz to {model_sr} Hz")
        resampler = torchaudio.transforms.Resample(orig_sr, model_sr)
        waveform = resampler(waveform)

    num_samples = waveform.shape[1]
    chunk_samples = int(chunk_length_sec * model_sr)
    overlap_samples = int(overlap_sec * model_sr)
    hop_samples = chunk_samples - overlap_samples

    # We'll accumulate separated sources in lists
    separated = []  # list of tensors, one per estimated source

    for start in range(0, num_samples, hop_samples):
        end = min(start + chunk_samples, num_samples)
        chunk = waveform[:, start:end].to("cuda")

        try:
            est_sources = separator.separate_batch(chunk)  # shape: (1, time, n_src)
            est_sources = est_sources.squeeze(0)  # (time, n_src)
            n_src = est_sources.shape[1]

            if len(separated) == 0:
                separated = [[] for _ in range(n_src)]

            for i in range(n_src):
                src_chunk = est_sources[:, i].cpu()
                if len(separated[i]) > 0:
                    # Overlap-add previous chunk's end with current start
                    prev = separated[i][-1]
                    fade_out = torch.hann_window(overlap_samples).to(prev.device)
                    fade_in = torch.hann_window(overlap_samples).to(src_chunk.device)
                    prev[-overlap_samples:] *= fade_out
                    src_chunk[:overlap_samples] *= fade_in
                    separated[i][-1] = prev
                separated[i].append(src_chunk)

            del est_sources, chunk
            torch.cuda.empty_cache()
            print(f"Processed chunk {start//hop_samples + 1} ({start/model_sr:.1f}s)")

        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"OOM on chunk starting at {start/model_sr:.1f}s → try smaller chunk_length_sec")
                raise
            else:
                raise

    # Concatenate with overlap already handled
    final_sources = []
    for src_list in separated:
        final = torch.cat(src_list, dim=0)
        final_sources.append(final)

    return final_sources, model_sr

# Process all files
for f in Path(in_path).iterdir():
    if f.suffix.lower() in ['.mp3', '.wav', '.flac', '.ogg', '.m4a']:
        print(f"\nSeparating → {f.name}")
        try:
            sources, sr = separate_with_chunks(f)
            out_folder = Path(out_path) / f.stem
            out_folder.mkdir(exist_ok=True)

            for i, src_tensor in enumerate(sources, 1):
                # Resample back to original sample rate if needed
                if sr != 48000:
                    resampler = torchaudio.transforms.Resample(sr, 48000)
                    src_tensor = resampler(src_tensor.unsqueeze(0)).squeeze(0)

                wav_tmp = out_folder / f"speaker_{i}.wav"
                sf.write(wav_tmp, src_tensor.numpy(), 48000)

                # Convert to MP3
                mp3_file = out_folder / f"speaker_{i}.mp3"
                sp.run([
                    "ffmpeg", "-y", "-i", str(wav_tmp),
                    "-b:a", "320k", "-ar", "48000", str(mp3_file)
                ], stdout=sp.DEVNULL, stderr=sp.DEVNULL)

                os.remove(wav_tmp)
                print(f" → Saved speaker_{i}.mp3 ({len(src_tensor)/48000:.1f}s)")

            print(f"✅ Finished {f.name}")

        except Exception as e:
            print(f"Failed {f.name}: {e}")

print("\n🎉 Done! Check voice_separated/ folder.")

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/sepformer_whamr16k_cache/hyperparams.yaml'
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/sepformer-whamr16k' if not cached


Loading Sepformer WHAMR! 16kHz model (2 speakers + noise/reverb)...


DEBUG:speechbrain.utils.parameter_transfer:Collecting files (or symlinks) for pretraining in sepformer_whamr16k_cache.
INFO:speechbrain.utils.fetching:Fetch masknet.ckpt: Using symlink found at '/content/sepformer_whamr16k_cache/masknet.ckpt'
DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["masknet"] = /content/sepformer_whamr16k_cache/masknet.ckpt
INFO:speechbrain.utils.fetching:Fetch encoder.ckpt: Using symlink found at '/content/sepformer_whamr16k_cache/encoder.ckpt'
DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["encoder"] = /content/sepformer_whamr16k_cache/encoder.ckpt
INFO:speechbrain.utils.fetching:Fetch decoder.ckpt: Using symlink found at '/content/sepformer_whamr16k_cache/decoder.ckpt'
DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["decoder"] = /content/sepformer_whamr16k_cache/decoder.ckpt
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: masknet, encoder, decoder
DEBUG:speechbra

Model loaded!

Separating → Guga_UlotraSeven_Sobreposto_Amarildo.mp3
Resampling from 48000 Hz to 16000 Hz
Processed chunk 1 (0.0s)
Processed chunk 2 (27.0s)
Processed chunk 3 (54.0s)
Processed chunk 4 (81.0s)
Processed chunk 5 (108.0s)
Processed chunk 6 (135.0s)
Processed chunk 7 (162.0s)
Processed chunk 8 (189.0s)
Processed chunk 9 (216.0s)
Processed chunk 10 (243.0s)
Processed chunk 11 (270.0s)
Processed chunk 12 (297.0s)
Processed chunk 13 (324.0s)
Processed chunk 14 (351.0s)
Processed chunk 15 (378.0s)
Processed chunk 16 (405.0s)
 → Saved speaker_1.mp3 (470.2s)
 → Saved speaker_2.mp3 (470.2s)
✅ Finished Guga_UlotraSeven_Sobreposto_Amarildo.mp3

🎉 Done! Check voice_separated/ folder.
